# Notebook 10 — A different idea: learn the entire CYP potency distribution

## An isoform-conditioned, threshold-based probability model

Instead of predicting one pIC50 directly, train **one shared classifier** to answer questions of the form: *For this molecule and CYP isoform, what is the probability that pIC50 exceeds threshold t?*

\[P(pIC_{50} > t\mid \mathrm{structure}, \mathrm{CYP})\]

Give the model molecular descriptors, Morgan fingerprints, the CYP identifier, and the continuously valued threshold. Apply a **monotonic constraint** so predicted exceedance probability cannot increase as the threshold rises. Integrate the exceedance curve to obtain an expected potency and invert it for estimated potency quantiles.

This is a **distributional / ordinal multi-isoform formulation**, substantially different from prior point-regression and auxiliary multitask models. It explicitly tests the longstanding range-compression and high-potency prediction problem. A second variant **ignores threshold decisions whose cutoff lies inside the reported experimental confidence interval**, rather than forcing those ambiguous decisions to 0 or 1.

**Baseline:** separate molecular LightGBM regressors on the exact same folds and features. An experimental distributional model is not presumed better; use these runs to learn whether it offers complementary prediction behavior or better activity-threshold decisions. CPU Colab is sufficient.

## 0 · Install packages

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.5" "scipy>=1.11" pandas matplotlib

## 1 · Configuration and imports

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, brier_score_loss
from scipy.stats import spearmanr
from scipy.integrate import trapezoid
import pandas as pd, numpy as np, lightgbm as lgb
import io, json, hashlib, zipfile, os, platform, time, warnings, random
import matplotlib.pyplot as plt
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'

def read_csv_source(filename, override=None):
    b=Path(override).read_bytes() if override else urlopen(URL+filename,timeout=180).read()
    return pd.read_csv(io.BytesIO(b)), hashlib.sha256(b).hexdigest()

def identity(smiles):
    mol=Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None
    if mol is None: return None,None
    canon=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(mol)
    scaffold=(Chem.MolToSmiles(core,canonical=True,isomericSmiles=False) if core.GetNumAtoms()
              else 'ACYCLIC_'+canon)
    return canon,scaffold

def assign_primary_folds(primary,n_splits=5):
    fold=np.full(len(primary),-1,dtype=int)
    for k,(_,val) in enumerate(GroupKFold(n_splits=n_splits).split(primary,groups=primary.scaffold_group)):
        fold[val]=k
    assert (fold>=0).all()
    assert not pd.DataFrame({'g':primary.scaffold_group,'f':fold}).groupby('g').f.nunique().gt(1).any()
    return fold

def read_zip_table(path,filename):
    with zipfile.ZipFile(path) as z:
        names=[n for n in z.namelist() if n==filename or n.endswith('/'+filename)]
        if len(names)!=1:raise ValueError(f'{filename}: ambiguous/missing in {path}: {names}')
        return pd.read_csv(io.BytesIO(z.read(names[0])))

def molecular_features(smiles, n_bits=512):
    molecules=[Chem.MolFromSmiles(s) for s in smiles]
    assert all(m is not None for m in molecules)
    desc=[]
    for m in molecules:
        with np.errstate(all='ignore'):
            desc.append([fn(m) for _,fn in Descriptors._descList])
    desc=np.asarray(desc,dtype=np.float32)
    desc[~np.isfinite(desc)]=np.nan
    generator=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=n_bits)
    fps=np.zeros((len(molecules),n_bits),dtype=np.float32)
    for i,m in enumerate(molecules):
        fp=generator.GetCountFingerprint(m)
        DataStructs.ConvertToNumpyArray(fp,fps[i])
    features=np.column_stack((desc,fps)).astype(np.float32)
    feature_names=[f'D_{i}_{name}' for i,(name,_) in enumerate(Descriptors._descList)]+[f'M_{i}' for i in range(n_bits)]
    return features,feature_names

def mae_per_iso(frame, pred_col):
    scores={}
    for iso,chunk in frame.groupby('isoform'):
        scores[iso]=float(np.mean(np.abs(chunk.y_true.to_numpy()-chunk[pred_col].to_numpy())))
    return scores

def macro_mae(frame,pred_col):
    vals=mae_per_iso(frame,pred_col)
    return float(np.mean([vals[k] for k in ISOFORMS if k in vals]))

def save_zip(out_dir, filename):
    path=Path(filename)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(Path(out_dir).rglob('*')):
            if p.is_file() and '_cache' not in p.parts:
                z.write(p,Path(out_dir).name+'/'+str(p.relative_to(out_dir)))
    return path

SEED=2026
FAST_MODE=True
N_SPLITS=5
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(5))
N_BITS=512
TREES_POINT=100 if FAST_MODE else 280
TREES_DISTRIBUTION=100 if FAST_MODE else 350
# This grid spans most of the observed pIC50 distribution. Endpoints are integration anchors.
THRESHOLDS=np.round(np.arange(2.25,7.76,0.5),2)
INTEGRATION_LO=1.8
INTEGRATION_HI=8.2
# CI-aware variant ignores targets whose credible interval contains the threshold.
RUN_INTERVAL_MASKED=True
# Experiment identifier must be changed if modeling choices change.
RUN_TAG='v1'
DATA_OVERRIDE=None  # '/content/my_train.csv' for offline synthetic tests
# Optional audit of Notebook 08 fold definitions:
NOTEBOOK08_RESULTS_ZIP=None # '/content/cyp_notebook_08_results.zip'
# To preserve outputs between Colab sessions:
# from google.colab import drive; drive.mount('/content/drive')
PERSISTENT_ROOT=None  # '/content/drive/MyDrive/cyp_notebook10'
BASE=Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT=BASE/'cyp_notebook_10_results';OUT.mkdir(parents=True,exist_ok=True)
print('Thresholds',THRESHOLDS,'range',INTEGRATION_LO,INTEGRATION_HI)

## 2 · Load data, audit scaffold splits and calculate molecular features

In [ ]:
direct,sha_direct=read_csv_source('cyp-challenge-TRAIN_inhibition.csv',DATA_OVERRIDE)
primary=direct.copy().reset_index(drop=True)
pairs=[identity(s) for s in primary.SMILES]
primary['canonical_smiles']=[a for a,b in pairs]
primary['scaffold_group']=[b for a,b in pairs]
assert primary.canonical_smiles.notna().all() and primary.canonical_smiles.is_unique
primary['fold']=assign_primary_folds(primary,N_SPLITS)
manifest=primary[['canonical_smiles','scaffold_group','fold']]
manifest.to_csv(OUT/'fold_manifest.csv',index=False)
if NOTEBOOK08_RESULTS_ZIP:
    previous=read_zip_table(NOTEBOOK08_RESULTS_ZIP,'fold_manifest.csv')
    current=manifest.set_index('canonical_smiles').fold
    old=previous.set_index('canonical_smiles').fold
    assert set(current.index)==set(old.index)
    assert np.array_equal(current.to_numpy(),old.loc[current.index].to_numpy()), 'Fold mismatch'
    print('Fold manifest matched previous Notebook 08')
Y=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(float) for iso in ISOFORMS])
LOW=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_low',np.nan),errors='coerce')
                     if f'{iso}_pIC50_direct_inhibition_conf_low' in primary else np.full(len(primary),np.nan)
                     for iso in ISOFORMS]).astype(float)
HIGH=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_high',np.nan),errors='coerce')
                      if f'{iso}_pIC50_direct_inhibition_conf_high' in primary else np.full(len(primary),np.nan)
                      for iso in ISOFORMS]).astype(float)
settings={'sha':sha_direct,'bits':N_BITS,'seed':SEED,'trees':TREES_DISTRIBUTION,'point_trees':TREES_POINT,
          'thresholds':THRESHOLDS.tolist(),'ci_mask':RUN_INTERVAL_MASKED,'tag':RUN_TAG}
RUN_ID=hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()[:16]
CACHE=BASE/('cyp_nb10_cache_'+RUN_ID);CACHE.mkdir(parents=True,exist_ok=True)
feature_cache=CACHE/'features.npz'
if feature_cache.exists():
    X=np.load(feature_cache)['X']
    assert X.shape[0]==len(primary)
else:
    X,_=molecular_features(primary.canonical_smiles.tolist(),N_BITS)
    np.savez_compressed(feature_cache,X=X)
print('Data:',len(primary),'features:',X.shape,'labels:',np.isfinite(Y).sum(axis=0))

## 3 · How the distributional model works

For each measured direct-inhibition pIC50, create binary labels for a grid of potency thresholds. The same structure appears at multiple thresholds, and all four CYPs share one model. The CYP is encoded one-hot.

We impose `monotone_constraints=-1` on **only the threshold column** of LightGBM. This ensures a higher threshold cannot produce a higher exceedance probability, while the model can learn arbitrary structure–CYP–threshold interactions.

The expected pIC50 is calculated by numerical integration over a **bounded** support. Predictions are consequently limited to the support [1.8, 8.2]. This choice must be justified and tested; do not claim that the distribution captures extrapolation beyond the bounds.

The interval-masked variant discards binary labels at thresholds inside the experimental credible interval; this is a **heuristic uncertainty-aware learning rule**, not a Bayesian posterior or the official challenge score. Because poorly constrained weak activities have wider intervals, this can change effective training weights substantially.

In [ ]:
def make_point(seed):
    return lgb.LGBMRegressor(n_estimators=TREES_POINT,learning_rate=.04,num_leaves=21,
        min_child_samples=10 if FAST_MODE else 20,colsample_bytree=.85,
        reg_lambda=3,verbosity=-1,n_jobs=2,random_state=seed)

def make_cdf(seed,nfeatures):
    constraint=[0]*(nfeatures-1)+[-1]
    return lgb.LGBMClassifier(n_estimators=TREES_DISTRIBUTION,learning_rate=.04,
        num_leaves=21,min_child_samples=20 if FAST_MODE else 40,
        reg_lambda=3,colsample_bytree=.90,
        monotone_constraints=constraint,verbosity=-1,n_jobs=2,random_state=seed)

def impute_numeric(train_idx):
    med=np.nanmedian(X[train_idx],axis=0)
    med[~np.isfinite(med)]=0.0
    return np.where(np.isfinite(X),X,med[None,:]).astype(np.float32)

def encode_rows(x, y, lo, hi, thresholds, mask_ci=False):
    """Return (features, classification target), one row per measured (molecule,CYP,t)."""
    valid=np.where(np.isfinite(y))
    ii,jj=valid
    n=len(ii)
    if n==0:raise ValueError('No training labels')
    idx=np.repeat(np.arange(n),len(thresholds))
    ti=np.tile(np.asarray(thresholds,dtype=np.float32),n)
    mi=ii[idx]; ji=jj[idx]
    onehot=np.eye(4,dtype=np.float32)[ji]
    feat=np.column_stack([x[mi],onehot,ti])
    labels=(y[mi,ji]>ti).astype(np.int8)
    if mask_ci:
        a=lo[mi,ji]; b=hi[mi,ji]
        # Exclude ambiguous thresholds only when confidence bounds are valid and contain t.
        ambiguous=np.isfinite(a)&np.isfinite(b)&(a<=ti)&(ti<=b)&(a<=b)
        feat=feat[~ambiguous]; labels=labels[~ambiguous]
    if len(np.unique(labels))<2:raise ValueError('Only one class after threshold encoding')
    return feat.astype(np.float32),labels

def predict_exceedance(model,eval_x,thresholds=THRESHOLDS):
    n=len(eval_x);k=len(thresholds)
    xx=np.repeat(eval_x,4*k,axis=0)
    ji=np.tile(np.repeat(np.arange(4),k),n)
    ti=np.tile(np.tile(thresholds,4),n)
    onehot=np.eye(4,dtype=np.float32)[ji]
    features=np.column_stack((xx,onehot,ti.astype(np.float32)))
    probs=model.predict_proba(features)[:,1].reshape(n,4,k)
    # Diagnostic safeguard, rather than primary monotonicity enforcement.
    violations=int(np.sum(np.diff(probs,axis=2)>1e-7))
    probs=np.minimum.accumulate(probs,axis=2)
    return probs,violations

def summarize_potency(probs,thresholds=THRESHOLDS):
    boundaries=np.r_[INTEGRATION_LO,thresholds,INTEGRATION_HI]
    survival=np.concatenate([np.ones((*probs.shape[:2],1)),probs,
                             np.zeros((*probs.shape[:2],1))],axis=2)
    # E[Y] = a + integral_a^b P(Y>t) dt, for Y bounded to [a,b].
    expectation=INTEGRATION_LO+trapezoid(survival,x=boundaries,axis=2)
    return expectation

def predicted_prob_at(probs,target=6.0):
    return np.array([np.interp(target,THRESHOLDS,p,left=1.,right=0.) for p in probs.reshape(-1,len(THRESHOLDS))]).reshape(probs.shape[:2])

## 4 · Fit common-feature point regressors and shared CDF classifiers

In [ ]:
all_rows=[];train_audit=[]
for fold in FOLDS_TO_RUN:
    path=CACHE/f'fold_{fold}.csv';audit_path=CACHE/f'fit_audit_fold_{fold}.csv'
    if path.exists() and audit_path.exists():
        print('Resuming',fold)
        all_rows.append(pd.read_csv(path));train_audit.append(pd.read_csv(audit_path));continue
    train=np.flatnonzero(primary.fold.to_numpy()!=fold)
    valid=np.flatnonzero(primary.fold.to_numpy()==fold)
    assert set(primary.scaffold_group.iloc[train]).isdisjoint(set(primary.scaffold_group.iloc[valid]))
    xx=impute_numeric(train)
    pred_point=np.full((len(valid),4),np.nan,dtype=np.float32)
    for j,iso in enumerate(ISOFORMS):
        observed=train[np.isfinite(Y[train,j])]
        model=make_point(SEED+fold*70+j)
        model.fit(xx[observed],Y[observed,j])
        pred_point[:,j]=model.predict(xx[valid])
    models={'CDF_hard':False}
    if RUN_INTERVAL_MASKED:models['CDF_interval_masked']=True
    preds={};classification_probs={};details=[]
    for name,mask in models.items():
        # Encode labels from outer TRAINING compounds exclusively; no validation-label access.
        xrows,yrows=encode_rows(xx[train],Y[train],LOW[train],HIGH[train],THRESHOLDS,mask)
        clf=make_cdf(SEED+fold*90+(3 if mask else 1),xrows.shape[1])
        clf.fit(xrows,yrows)
        p,violations=predict_exceedance(clf,xx[valid])
        preds[name]=summarize_potency(p)
        classification_probs[name]=predicted_prob_at(p,6.0)
        details.append({'fold':fold,'model':name,'n_training_binary_rows':len(yrows),
                        'positive_fraction':float(np.mean(yrows)),
                        'monotonicity_violations_before_projection':violations,
                        'train_scaffold_overlap':0})
    records=[]
    for pos,idx in enumerate(valid):
        for j,iso in enumerate(ISOFORMS):
            if not np.isfinite(Y[idx,j]):continue
            row={'canonical_smiles':primary.canonical_smiles.iloc[idx],
                 'scaffold_group':primary.scaffold_group.iloc[idx],
                 'fold':fold,'isoform':iso,'y_true':float(Y[idx,j]),
                 'ci_low':LOW[idx,j],'ci_high':HIGH[idx,j],
                 'Point_LGBM':float(pred_point[pos,j])}
            for name in preds:
                row[name]=float(preds[name][pos,j]);row[f'{name}_P_gt6']=float(classification_probs[name][pos,j])
            records.append(row)
    df_fold=pd.DataFrame(records)
    audit_fold=pd.DataFrame(details)
    for df,path2 in [(df_fold,path),(audit_fold,audit_path)]:
        tmp=path2.with_suffix('.tmp');df.to_csv(tmp,index=False);os.replace(tmp,path2)
    all_rows.append(df_fold);train_audit.append(audit_fold)
    print('Fold',fold,'complete, n measured predictions',len(df_fold))
preds=pd.concat(all_rows,ignore_index=True)
audit=pd.concat(train_audit,ignore_index=True)
assert not preds.duplicated(['isoform','canonical_smiles']).any()
assert (audit.train_scaffold_overlap==0).all()
preds.to_csv(OUT/'oof_predictions.csv',index=False)
audit.to_csv(OUT/'training_audit.csv',index=False)
print('Completed',len(preds),'observations; folds',sorted(preds.fold.unique()))

## 5 · Evaluate potency MAE, bias, rank correlation and predicted-range compression

In [ ]:
model_cols=['Point_LGBM','CDF_hard']+(['CDF_interval_masked'] if RUN_INTERVAL_MASKED else [])
rows=[];tails=[];threshold_metrics=[]
for iso,f in preds.groupby('isoform'):
    lo,hi=f.y_true.quantile([.1,.9]).to_numpy()
    for model in model_cols:
        obs=f.y_true.to_numpy();pred=f[model].to_numpy()
        rho=spearmanr(obs,pred).statistic if len(obs)>2 else np.nan
        rows.append({'isoform':iso,'model':model,'n':len(f),'mae':float(np.mean(np.abs(obs-pred))),
                     'r2':float(r2_score(obs,pred)),'spearman':float(rho),
                     'bias':float(np.mean(pred-obs)),
                     'pred_over_true_sd':float(np.std(pred)/np.std(obs))})
        for part,mask in [('low_10pct',obs<=lo),('middle_80pct',(obs>lo)&(obs<hi)),('high_10pct',obs>=hi)]:
            tails.append({'isoform':iso,'model':model,'subset':part,'n':int(mask.sum()),
                          'mae':float(np.mean(np.abs(obs[mask]-pred[mask]))),
                          'signed_bias':float(np.mean(pred[mask]-obs[mask]))})
        if model.startswith('CDF_'):
            true=(obs>6.).astype(int)
            prob=f[f'{model}_P_gt6'].to_numpy()
            threshold_metrics.append({'isoform':iso,'model':model,'threshold':6.,'n_positive':int(true.sum()),
               'brier':float(brier_score_loss(true,prob)),
               'roc_auc':float(roc_auc_score(true,prob)) if len(np.unique(true))==2 else np.nan})
metrics_df=pd.DataFrame(rows)
tails_df=pd.DataFrame(tails)
threshold_df=pd.DataFrame(threshold_metrics)
metrics_df.to_csv(OUT/'model_metrics.csv',index=False)
tails_df.to_csv(OUT/'activity_extremes.csv',index=False)
threshold_df.to_csv(OUT/'threshold_probability_metrics.csv',index=False)
macro=metrics_df.groupby('model').mae.mean().sort_values()
display(metrics_df.pivot(index='isoform',columns='model',values='mae'))
display(macro)
display(tails_df.pivot_table(index=['isoform','subset'],columns='model',values='mae'))
display(threshold_df)

## 6 · Visualize experimental-range compression and extreme-potency errors

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,5))
for model in model_cols:
    subset=metrics_df[metrics_df.model==model]
    axs[0].plot(subset.isoform,subset.pred_over_true_sd,marker='o',label=model)
axs[0].axhline(1,color='black',lw=0.8,ls='--')
axs[0].set_ylabel('SD(predicted) / SD(measured)')
axs[0].set_title('Prediction-range compression')
axs[0].legend(fontsize=8)
for model in model_cols:
    subset=tails_df[(tails_df.model==model)&(tails_df.subset=='high_10pct')]
    axs[1].plot(subset.isoform,subset.mae,marker='o',label=model)
axs[1].set_title('High-potency extreme: highest pIC50 decile')
axs[1].set_ylabel('MAE (pIC50)');axs[1].legend(fontsize=8)
plt.tight_layout();plt.savefig(OUT/'01_distribution_model_results.png',dpi=160);plt.show()

## 7 · Check ensemble complementarity without optimizing weights on the same labels

For **diagnostic purposes only**, compare a *pre-specified* equal-weight average of the point-regression and distributional model. If this appears helpful, evaluate it with a fresh group holdout or nested selection later. This notebook does not retrospectively optimize weights to claim an improvement.

In [ ]:
preds['fixed_equal_point_cdf']=.5*preds.Point_LGBM+.5*preds.CDF_hard
complement=[]
for iso,f in preds.groupby('isoform'):
    complement.append({'isoform':iso,'point_mae':float(np.mean(np.abs(f.y_true-f.Point_LGBM))),
           'cdf_mae':float(np.mean(np.abs(f.y_true-f.CDF_hard))),
           'fixed_50_50_mae':float(np.mean(np.abs(f.y_true-f.fixed_equal_point_cdf))),
           'residual_pearson':float(np.corrcoef(f.y_true-f.Point_LGBM,f.y_true-f.CDF_hard)[0,1])})
complement_df=pd.DataFrame(complement)
complement_df.to_csv(OUT/'fixed_blend_diagnostic.csv',index=False)
preds.to_csv(OUT/'oof_predictions.csv',index=False)
display(complement_df)

## 8 · Export results and record limitations

In [ ]:
metadata={'notebook':'10_isoform_conditioned_potency_distribution.ipynb',
          'source_sha256':sha_direct,'run_id':RUN_ID,
          'folds_completed':sorted(preds.fold.unique().astype(int).tolist()),
          'fast_mode':FAST_MODE,'n_bits':N_BITS,
          'thresholds':THRESHOLDS.tolist(),'integration_bounds':[INTEGRATION_LO,INTEGRATION_HI],
          'interval_masked':RUN_INTERVAL_MASKED,
          'note':'Experimental isoform-conditioned ordinal CDF. MAE is not the official MA-ST-RAE.'}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
archive=save_zip(OUT,'cyp_notebook_10_results.zip')
print('Created:',archive,'bytes',archive.stat().st_size)

## Discussion and research interpretation

1. Does the distributional classifier predict high-inhibition potency more accurately, or does it merely improve classification at a chosen threshold?
2. Does the predicted/true pIC50 standard-deviation ratio approach 1 without making overall MAE worse?
3. Does confidence-interval threshold masking help at low activity but impair high-potency separation?
4. Are threshold probabilities reasonably calibrated? The provided Brier score is useful but should be complemented by reliability curves before reporting calibrated risk.
5. Are the distributional model errors complementary to baseline regression?

**Scope:** Quantile/exceedance predictions are bounded by the integration range, the interval-mask rule is heuristic, and the shared LightGBM architecture may underperform trained CheMeleon. This is an explicitly exploratory hypothesis, not an assumed improvement. Unlike the prior screening-transfer models, it uses **direct-inhibition labels only** and requires no new protein structure or proprietary data.